# Enterprise Knowledge Assistant (Capstone)

This notebook integrates the core components of the Beginner track into a realistic Enterprise RAG system. We will evaluate how different **retrieval strategies** (Base, Multi-Query Expansion, and Cross-Encoder Reranking) impact retrieval and generation metrics across a synthetic corpus.


In [ ]:
import os
import json
import time
from dotenv import load_dotenv

# Ensure we have our environment variables (e.g., OPENAI_API_KEY)
load_dotenv()

# Import from our application modules
from src.config import TOP_K, CHUNK_SIZE, CHUNK_OVERLAP
from src.ingestion import load_knowledge_base
from src.chunking import chunk_documents_structured
from src.retrieval import get_vector_store, index_documents, retrieve_base, retrieve_expanded, retrieve_reranked, retrieve_filtered
from src.generation import build_evidence_context, generate_decision
from src.validation import validate_decision, render_response


## 1. Ingestion and Chunking

We will ingest our synthetic enterprise knowledge base and chunk it using our Structure-Aware Markdown chunker.

In [ ]:
print("Loading documents from data/knowledge_base...")
docs = load_knowledge_base("data/knowledge_base")
print(f"Loaded {len(docs)} documents.\n")

print("Chunking: Structure-Aware Markdown")
chunks_structured = chunk_documents_structured(docs, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
print(f"Created {len(chunks_structured)} chunks.")


## 2. Vector Indexing

We will index the chunks into a Chroma vector store collection.

In [ ]:
# Build the indexes (this takes a moment to compute embeddings)
print("Indexing Structured...")
store_structured = get_vector_store("./chroma_db", "kb_structured")
# index_documents(chunks_structured, "./chroma_db", "kb_structured") # Uncomment to rebuild

print("Vector store ready.")


## 3. The RAG Pipeline with Pluggable Retrieval

Let's define our core orchestration function. We've updated it to accept a `retrieval_strategy` parameter so we can test different approaches.

In [ ]:
def run_enterprise_rag(query: str, vector_store, top_k: int = TOP_K, retrieval_strategy="base", render: bool = True):
    start_time = time.time()
    
    # 1. Retrieval
    if retrieval_strategy == "base":
        candidates = retrieve_base(query, vector_store, top_k)
    elif retrieval_strategy == "expanded":
        candidates = retrieve_expanded(query, vector_store, top_k)
    elif retrieval_strategy == "reranked":
        expanded_candidates = retrieve_expanded(query, vector_store, top_k)
        candidates = retrieve_reranked(query, expanded_candidates, top_k)
    elif retrieval_strategy == "filtered":
        candidates = retrieve_filtered(query, vector_store, top_k)
    else:
        raise ValueError("Invalid strategy")
    
    # 2. Context Building
    context_str, evidence_map = build_evidence_context(candidates)
    
    # 3. Generation
    decision = generate_decision(query, context_str)
    
    # 4. Validation
    errors = validate_decision(decision, evidence_map)
    if errors and render:
        print(f"\n[VALIDATION FAILED] \n" + "\n".join(errors))
        
    # 5. Rendering
    if render:
        render_response(decision, evidence_map)
        latency = time.time() - start_time
        print(f"\n(Query latency: {latency:.2f}s)")
        
    return candidates, evidence_map, decision, errors


## 4. Advanced Retrieval Walkthrough

Let's see how our retrieval improves step-by-step on a difficult query: **"How often do P1 customers get updates?"**

In [ ]:
query = "How often do P1 customers get updates?"
print("--- 1. Base Retrieval ---")
base_docs = retrieve_base(query, store_structured, TOP_K)
for i, d in enumerate(base_docs):
    print(f"{i+1}. {d.metadata.get('document_id')} (Section: {d.metadata.get('section', 'N/A')})")

print("\n--- 2. Expanded Retrieval (Multi-Query) ---")
expanded_docs = retrieve_expanded(query, store_structured, TOP_K)
for i, d in enumerate(expanded_docs):
    print(f"{i+1}. {d.metadata.get('document_id')} (Section: {d.metadata.get('section', 'N/A')})")

print("\n--- 3. Cross-Encoder Reranked ---")
reranked_docs = retrieve_reranked(query, expanded_docs, TOP_K)
for i, d in enumerate(reranked_docs):
    print(f"{i+1}. {d.metadata.get('document_id')} (Section: {d.metadata.get('section', 'N/A')})")


## 5. Metadata Filtering (Pre-Retrieval)

Sometimes, we know exactly which department a question pertains to. We can use the LLM to extract this intent and apply it as a strict metadata filter before semantic search even begins.

In [ ]:
query = "What is the parental leave policy in HR?"
print("--- Filtered Retrieval ---")
filtered_docs = retrieve_filtered(query, store_structured, TOP_K)
for i, d in enumerate(filtered_docs):
    print(f"{i+1}. {d.metadata.get('document_id')} (Department: {d.metadata.get('department')})")


## 6. End-to-End Comparative Evaluation

Let's run our expanded golden dataset across all three retrieval strategies using the structured chunk store. We will capture `Recall@K`, `Context Precision (MRR)`, `Abstention Accuracy`, `Citation Validity`, `Semantic Correctness`, and `Strict Groundedness`.

In [ ]:
# Load the golden dataset
with open("data/evaluation/golden_dataset.json", "r") as f:
    eval_dataset = json.load(f)

from src.evaluation import calculate_recall_at_k, calculate_mrr, evaluate_abstention, evaluate_correctness, evaluate_groundedness

def evaluate_retrieval_strategy(strategy_name, dataset):
    print(f"\nEvaluating {strategy_name}...")
    total_queries = len(dataset)
    successful_recalls = 0
    mrr_sum = 0
    valid_citations = 0
    correctness_score = 0
    groundedness_score = 0
    abstention_stats = {"True Answer": 0, "True Abstention": 0, "False Abstention": 0, "Unsafe Answer": 0, "Unknown": 0}

    for item in dataset:
        # We disable rendering for the eval run
        candidates, evidence_map, decision, errors = run_enterprise_rag(
            item["question"], store_structured, top_k=TOP_K, retrieval_strategy=strategy_name, render=False
        )
        
        # Deterministic metrics
        if item["answerable"]:
            successful_recalls += calculate_recall_at_k(candidates, item["expected_document_ids"])
            mrr_sum += calculate_mrr(candidates, item["expected_document_ids"])
            
        if not errors and decision.decision == "answer":
            valid_citations += 1
            
        # Abstention Accuracy
        abstention_class = evaluate_abstention(decision.decision, item["answerable"])
        abstention_stats[abstention_class] += 1
        
        # LLM-as-Judge
        if decision.decision == "answer" and item["answerable"]:
            is_correct = evaluate_correctness(decision.answer, item["expected_answer"])
            if is_correct: correctness_score += 1
                
            is_grounded = evaluate_groundedness(decision.answer, decision.citations, evidence_map)
            if is_grounded: groundedness_score += 1

    answerable_queries = sum(1 for item in dataset if item["answerable"])
    answered_queries = abstention_stats["True Answer"] + abstention_stats["Unsafe Answer"]
    
    return {
        "Strategy": strategy_name,
        "Recall": (successful_recalls/answerable_queries)*100,
        "MRR": (mrr_sum/answerable_queries),
        "Abstention": ((abstention_stats['True Answer'] + abstention_stats['True Abstention'])/total_queries)*100,
        "Citation": (valid_citations/answered_queries)*100 if answered_queries > 0 else 0,
        "Correctness": (correctness_score/answered_queries)*100 if answered_queries > 0 else 0,
        "Groundedness": (groundedness_score/answered_queries)*100 if answered_queries > 0 else 0
    }

results = []
results.append(evaluate_retrieval_strategy("base", eval_dataset))
results.append(evaluate_retrieval_strategy("expanded", eval_dataset))
results.append(evaluate_retrieval_strategy("reranked", eval_dataset))
results.append(evaluate_retrieval_strategy("filtered", eval_dataset))

import pandas as pd
df = pd.DataFrame(results)
display(df)
